# STITCHV2 JupyterLab Tutorial

This notebook is a practical, end-to-end guide to STITCHV2. It explains input formats, every `stitchv2 run` parameter, original STITCH equivalents, BAM/PLINK/parquet workflows, pedigrees, mixed ploidy, scaling, outputs, and conversion to PLINK-friendly formats.

The examples are written so you can run them from the repository root:

```bash
cd /home/bonnie/Documents/codex/STITCHV2
conda activate /home/bonnie/Documents/codex/STITCHV2/.conda-benchmark-unified
export PYTHONPATH=/home/bonnie/Documents/codex/STITCHV2/src
jupyter lab
```

Most cells are safe to read without executing. Cells that run full benchmarks are marked explicitly.


## 0. Mental Model: What STITCHV2 Does

STITCHV2 imputes genotypes from low-coverage sequencing reads, optional array genotypes, and founder haplotype states.

At a high level:

1. Read the sample table and position table.
2. Extract read evidence from BAM/CRAM files by variant block.
3. Optionally inject PLINK microarray hard calls as strong genotype evidence.
4. Run a read-aware HMM over founders.
5. Optionally update mutable founders by EM.
6. Calibrate genotype posteriors and emit dosage, genotype calls, posteriors, transitions, recombination rates, and founder updates.
7. Keep primary results in Parquet/Zarr; export BCF only when an external tool explicitly requires it.

Original STITCH does similar imputation, but its public workflow is centered on `bamlist.txt`, `posfile`, `K`, `nGen`, `niterations`, and VCF outputs. STITCHV2 uses explicit parquet tables and Python/JAX/Dask backends so large jobs can be inspected and chunked more directly.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

REPO = Path('/home/bonnie/Documents/codex/STITCHV2')
DATA_DIR = REPO / 'benchmark_runs' / 'synth_5mb_2k_0p1x'
OUT_BASE = REPO / 'benchmark_runs' / 'tutorial_examples'
OUT_BASE.mkdir(parents=True, exist_ok=True)

print('Repository:', REPO)
print('Synthetic data exists:', DATA_DIR.exists())


## 1. Input Formats

STITCHV2 accepts tables in parquet or CSV/TSV-like text for samples and positions. BAM/CRAM paths, PLINK prefixes, founder VCF/PLINK files, and optional pedigree matrices are layered on top.


### 1.1 Sample Table: `samples.parquet` or CSV/TSV

Required logical columns:

| Column | Required? | Type | Meaning | STITCH equivalent |
|---|---:|---|---|---|
| `sample_id` | recommended | string | Individual ID. If missing, STITCHV2 creates `sample_0`, `sample_1`, ... | `sampleNames_file` |
| `bam_path` | filled if missing | string | Path to BAM/CRAM. Empty string means no reads for that sample. | `bamlist.txt` |
| `generation` | yes | float | Generations or recombination scaling parameter for transitions. | `nGen`, but STITCHV2 can vary by sample |
| `sex` | optional | string | Used with `--ploidy-males/--ploidy-females`; accepts values like `M`, `F`, `1`, `2`, `XY`, `XX`. | no direct equivalent |
| `plink_path` | optional | string | Per-sample PLINK prefix used to inject hard microarray calls. Multiple rows may point to the same prefix. | no direct direct equivalent; closest is external genotype/reference evidence |

A sample can have reads, PLINK array data, both, or neither. If both reads and array genotypes are present, STITCHV2 combines them by adding strong hard-call evidence from the array.


In [ ]:
# Example sample table with reads only
samples_reads = pd.DataFrame({
    'sample_id': ['animal_001', 'animal_002'],
    'bam_path': ['/data/bams/animal_001.bam', '/data/bams/animal_002.bam'],
    'generation': [10.0, 10.0],
})
samples_reads


In [ ]:
# Example mixed sample table: reads + per-sample PLINK microarray evidence + sex labels
samples_mixed = pd.DataFrame({
    'sample_id': ['animal_001', 'animal_002', 'array_only_003'],
    'bam_path': ['/data/bams/animal_001.bam', '/data/bams/animal_002.bam', ''],
    'generation': [10.0, 10.0, 10.0],
    'sex': ['M', 'F', 'F'],
    'plink_path': ['/data/arrays/batch_A', '/data/arrays/batch_A', '/data/arrays/batch_B'],
})
samples_mixed


### 1.2 Position Table: `positions.parquet` or CSV/TSV

Required columns:

| Column | Type | Meaning | STITCH equivalent |
|---|---|---|---|
| `CHR` | string | Chromosome/contig label. | `posfile` chromosome column or `chr` argument |
| `POS` | int | 1-based genomic coordinate. | `posfile` position |
| `REF` | string | Reference allele. | `posfile` ref allele |
| `ALT` | string | Alternate allele. | `posfile` alt allele |

Positions are filtered by `--chromosome`, optionally by `--chr-start` and `--chr-end`, then sorted by `POS`.


In [ ]:
positions_example = pd.DataFrame({
    'CHR': ['chr1', 'chr1', 'chr1'],
    'POS': [100123, 100456, 101000],
    'REF': ['A', 'C', 'G'],
    'ALT': ['G', 'T', 'A'],
})
positions_example


### 1.3 BAM/CRAM Inputs

BAM/CRAM files are referenced from `samples.bam_path`. They should be coordinate-sorted and indexed (`.bai` for BAM, `.crai` for CRAM). The contig names must match the `CHR` labels in `positions.parquet` and `--chromosome`.

Useful checks:

```bash
samtools quickcheck sample.bam
samtools index sample.bam
samtools idxstats sample.bam | head
```

STITCH equivalent: original STITCH uses a `bamlist.txt` file, one BAM path per line. STITCHV2 stores the same information in `samples.parquet` so sample metadata, sex, generation, and array paths travel together.


In [ ]:
# Inspect the synthetic data's sample table.
# This dataset was generated with tiny BAM files for tutorial/benchmark use.
if DATA_DIR.exists():
    display(pd.read_parquet(DATA_DIR / 'samples.parquet').head())
    display(pd.read_parquet(DATA_DIR / 'positions.parquet').head())


### 1.4 PLINK Inputs: Founder Panels and Microarray Evidence

STITCHV2 uses PLINK BED/BIM/FAM prefixes in two different ways:

1. **Founder initialization** with `--founder-plink /path/to/prefix`.
2. **Microarray hard-call evidence** with `--microarray-plink /path/to/prefix` or `samples.parquet: plink_path`.

PLINK prefix means these files exist:

```text
/path/to/prefix.bed
/path/to/prefix.bim
/path/to/prefix.fam
```

For per-sample `plink_path`, STITCHV2 loads each unique prefix once, extracts only the rows matching `sample_id`, and aligns variants by chromosome and position.


In [ ]:
# Example: inspect PLINK metadata lazily with STITCHV2's npplink helper.
# Change prefix to a real PLINK prefix before running.
RUN_PLINK_EXAMPLE = False
if RUN_PLINK_EXAMPLE:
    from stitchv2.npplink import read_fam_bim, load_plink_xarray
    prefix = Path('/data/arrays/batch_A')
    fam, bim = read_fam_bim(prefix)
    display(fam.head())
    display(bim.head())
    geno = load_plink_xarray(prefix, chunk_variants=10_000)
    print(geno)


### 1.5 Founder Inputs

STITCHV2 can use:

| Founder source | STITCHV2 input | STITCH equivalent | When to use |
|---|---|---|---|
| Uniform mutable founders | no founder file | STITCH estimates ancestral haplotypes internally | Exploratory runs, no reference founders |
| Founder VCF | `--founder-vcf founders.vcf.gz` | reference haplotype/legend/sample files, closest | Parity or known-founder synthetic experiments |
| Founder PLINK | `--founder-plink founders_prefix` | reference haplotypes, closest | When founder genotypes are already in BED/BIM/FAM |
| In-memory `FounderPanel` | Python API | no direct equivalent | Tests, custom pipelines, synthetic truth |

Use `--founder-immutable` to freeze founder states. This is the closest STITCH-parity mode when you have known hard haplotypes.


### 1.6 Optional Pedigree Input

The public CLI currently exposes `--pedigree-strength`, but pedigree data itself is passed through the Python API as a `scipy.sparse.csr_matrix` to `pipeline.prepare_inputs(samples, pedigree=pedigree)`.

Expected shape: `(n_samples, n_samples)`.

Interpretation: row `i` lists parent/relative contributors for sample `i`. After HMM dosage inference, STITCHV2 computes a parent/relative mean dosage and blends:

```text
smoothed_dosage = (1 - pedigree_strength) * HMM_dosage + pedigree_strength * parent_mean_dosage
```

STITCH equivalent: no direct command-line equivalent in standard STITCH. This is a STITCHV2 extension.


In [ ]:
# Example pedigree matrix: child sample 2 is smoothed toward samples 0 and 1.
from scipy import sparse

n_samples = 3
rows = [2, 2]       # child row
cols = [0, 1]       # parent columns
values = [1.0, 1.0]
pedigree = sparse.csr_matrix((values, (rows, cols)), shape=(n_samples, n_samples))
pedigree.toarray()


## 2. Every `stitchv2 run` Parameter and Closest STITCH Equivalent

This table focuses on `stitchv2 run`, the main imputation command. Parameters with no STITCH equivalent are STITCHV2-specific features such as JAX, Dask, parquet output, posterior calibration, microarray injection, and mixed ploidy.


| STITCHV2 parameter | Example | Closest STITCH equivalent | What it controls |
|---|---|---|---|
| `--samples` | `benchmark_runs/synth_5mb_2k_0p1x/samples.parquet` | `bamlist + sampleNames_file` | Sample metadata table. Required columns: generation; sample_id and bam_path are filled/normalized if absent. STITCH uses bamlist.txt and optional sampleNames_file instead of one table. |
| `--positions` | `benchmark_runs/synth_5mb_2k_0p1x/positions.parquet` | `posfile` | Variant table with CHR, POS, REF, ALT. CSV/TSV and parquet are accepted. |
| `--chromosome` | `chrSynthetic` | `chr` | Chromosome/contig to process. Must match positions and BAM/VCF contig labels. |
| `--chr-start` | `100000` | `no direct equivalent` | Inclusive coordinate window start. Use with --chr-end to run one chromosome chunk. |
| `--chr-end` | `200000` | `no direct equivalent` | Inclusive coordinate window end. Useful for smoke tests or sharded chromosome runs. |
| `--output-dir` | `benchmark_runs/tutorial_run` | `outputdir` | Run directory. STITCHV2 writes parquet chunks plus summaries; STITCH writes VCF/RData/tmp outputs. |
| `--n-founders` | `8` | `K` | Number of ancestral founder states. Larger K can improve diversity but increases state/emission work. |
| `--ploidy` | `2` | `method, closest concept` | Default sample ploidy. 0 is allowed and returns all-missing outputs without HMM compute. |
| `--ploidy-males` | `1` | `no direct equivalent` | Male ploidy override using samples.sex. Example: chrX male ploidy 1. |
| `--ploidy-females` | `2` | `no direct equivalent` | Female ploidy override using samples.sex. Example: chrX female ploidy 2; chrY female ploidy 0. |
| `--block-size` | `1000` | `outputBlockSize, closest concept` | Number of variants per STITCHV2 processing block. Larger blocks reduce overhead and increase memory. |
| `--em-iterations` | `5` | `niterations` | EM/founder-update iterations. For parity/accuracy, 5-10 is usually safer than 1-2. |
| `--hmm-backend` | `jax` | `no direct equivalent` | STITCHV2 compute backend: auto, numpy, jax, torch. Original STITCH uses its R/C++ backend. |
| `--jax-sample-batch-size` | `128` | `no direct equivalent` | Samples per JAX forward-backward batch. Lower reduces peak device memory; 0 means all samples. |
| `--executor` | `dask` | `no direct equivalent` | serial or dask orchestration. Dask schedules coarse JAX leaf tasks; it does not replace HMM math. |
| `--dask-scheduler` | `local` | `no direct equivalent` | Currently local scheduler wrapper for STITCHV2 Dask execution. |
| `--dask-n-workers` | `4` | `nCores, closest concept` | Number of local Dask workers. Keep coarse chunks to avoid scheduler overhead. |
| `--dask-threads-per-worker` | `1` | `nCores, closest concept` | Threads per worker. For JAX/GPU, 1 thread per worker is often easier to reason about. |
| `--dask-processes` | `--dask-processes` | `no direct equivalent` | Use process workers instead of thread workers. Useful for CPU isolation; more serialization cost. |
| `--dask-memory-limit` | `16GB` | `no direct equivalent` | Per-worker Dask memory limit. |
| `--dask-dashboard-address` | `127.0.0.1:8786` | `no direct equivalent` | Dashboard bind address for live scheduler diagnostics. |
| `--dask-performance-report` | `dask_report.html` | `no direct equivalent` | Permanent Dask HTML performance report. |
| `--dask-task-stream` | `task_stream.json` | `no direct equivalent` | Captured task stream as JSON/HTML. |
| `--dask-dashboard-hold-seconds` | `60` | `no direct equivalent` | Keep dashboard alive briefly after compute before cluster shutdown. |
| `--dask-target-task-memory-mb` | `4096` | `no direct equivalent` | Planner target for rough HMM task memory. If set, block/sample chunks are reduced conservatively. |
| `--dask-min-block-size` | `128` | `no direct equivalent` | Lower bound for Dask-planned SNP block size to avoid tiny tasks. |
| `--dask-min-sample-batch-size` | `8` | `no direct equivalent` | Lower bound for Dask-planned sample batch size. |
| `--dask-sample-batch-size` | `64` | `no direct equivalent` | Samples per Dask HMM task. Distinct from JAX-internal batching. |
| `--read-mode` | `read_stream` | `readAware=TRUE, closest concept` | read_stream is the primary read-aware path; pileup is a simpler alternative. |
| `--read-stream-backend` | `auto` | `no direct equivalent` | auto/htslib/python read extraction backend. htslib is fastest when compiled. |
| `--io-workers` | `8` | `nCores, closest concept` | Parallel read extraction across samples. |
| `--htslib-threads-per-file` | `2` | `nCores, closest concept` | HTSlib decompression threads per BAM/CRAM file. |
| `--fragment-likelihood-mode` | `augment` | `read-aware emission internals` | augment adds fragment likelihood to per-site emissions; replace substitutes centered per-site terms. |
| `--fragment-coupling-model` | `stitch_parity` | `readAware=TRUE internals` | STITCH-aligned fragment/read coupling mode. |
| `--fragment-max-diff-reads` | `100.0` | `maxDifferenceBetweenReads` | Clips extreme multi-read likelihood differences for numerical stability. |
| `--fragment-max-emission-diff` | `1000.0` | `maxEmissionMatrixDifference` | Clips extreme emission matrix differences. |
| `--no-fragment-rescale-read-likelihood` | `flag` | `no direct equivalent` | Disables STITCHV2 read-likelihood rescaling. Leave off for stable default behavior. |
| `--write-transitions` | `flag` | `no direct equivalent` | Write compact transition probabilities per sample/variant. |
| `--write-haplotype-probabilities` | `flag` | `output_haplotype_dosages` | Write founder haplotype probability/dosage vectors. |
| `--write-genotype-posteriors` | `flag` | `GP output, closest` | Write genotype posterior fixed-size-list parquet. |
| `--write-genotype-calls` | `flag` | `GT output` | Write hard genotype alt-count calls; -1 means no-call. |
| `--write-support-mask` | `flag` | `no direct equivalent` | Write whether each sample/SNP had direct read or microarray support. |
| `--no-calibrate-genotype-posteriors` | `flag` | `no direct equivalent` | Disable STITCHV2 posterior calibration. Use for strict parity diagnostics. |
| `--genotype-posterior-temperature` | `0.35` | `no direct equivalent` | Controls dosage-to-GP sharpening during calibration/fallback posterior construction. |
| `--genotype-posterior-blend` | `0.35` | `no direct equivalent` | Blends calibrated dosage-derived posterior with raw posterior. |
| `--genotype-call-mode` | `stitch_no_call` | `STITCH GP thresholding, closest` | argmax always calls; stitch_no_call applies GP threshold; quality_gated adds confidence/margin filters. |
| `--genotype-call-min-confidence` | `0.8` | `no direct equivalent` | Minimum top posterior probability for quality_gated calls. |
| `--genotype-call-min-margin` | `0.2` | `no direct equivalent` | Minimum gap between top two genotype posterior probabilities. |
| `--genotype-call-stitch-threshold` | `0.9` | `STITCH no-call threshold, closest` | Threshold used by stitch_no_call mode. |
| `--genotype-call-correctness-threshold` | `0.8` | `no direct equivalent` | Optional learned call-correctness threshold for quality_gated mode. |
| `--use-lightgbm-calibrator` | `flag` | `no direct equivalent` | Use LightGBM/isotonic calibration when microarray truth is available. |
| `--calibration-context-window` | `25` | `no direct equivalent` | Variant context window for calibration features. |
| `--calibration-block-snps` | `64` | `no direct equivalent` | Calibration chunking over SNPs. |
| `--calibration-use-optuna` | `flag` | `no direct equivalent` | Tune calibration model hyperparameters with Optuna. |
| `--calibration-optuna-trials` | `20` | `no direct equivalent` | Number of Optuna trials. |
| `--calibration-max-train-rows` | `750000` | `no direct equivalent` | Cap on calibration training rows. |
| `--microarray-plink` | `/data/array/genotypes` | `genfile/reference input, closest` | Global PLINK prefix with hard genotype evidence. Can add samples not in BAM table. |
| `--microarray-generation-default` | `10` | `no direct equivalent` | Generation value assigned to PLINK-only samples added by --microarray-plink. |
| `--microarray-hard-call-weight` | `80` | `no direct equivalent` | Pseudo-read evidence weight injected for each hard array call. |
| `--no-microarray-add-samples` | `flag` | `no direct equivalent` | Do not append array-only samples from global PLINK input. |
| `--random-seed` | `7` | `seed in scripts` | Controls jitter/subsampling choices where applicable. |
| `--founder-init-jitter` | `0.01` | `no direct equivalent` | Small random perturbation for mutable founder initialization. |
| `--memory-map-read-matrices` | `flag` | `no direct equivalent` | Store read matrices as memmaps to reduce peak RAM pressure. |
| `--memory-map-dir` | `/tmp/stitchv2_memmap` | `no direct equivalent` | Directory for read-matrix memmap files. |
| `--no-profile-memory` | `flag` | `no direct equivalent` | Disable per-block RSS profiling. |
| `--compression` | `zstd` | `no direct equivalent` | Parquet compression codec. |
| `--compression-level` | `6` | `no direct equivalent` | Parquet compression level. |
| `--pedigree-strength` | `0.1` | `no direct equivalent` | Strength for pedigree dosage smoothing if pedigree matrix is supplied through Python API. |
| `--founder-vcf` | `founders.truth.vcf.gz` | `reference_haplotype_file/reference_legend_file, closest` | Initialize founders from VCF. |
| `--founder-plink` | `/data/founders` | `reference haplotype inputs, closest` | Initialize founders from PLINK BED/BIM/FAM prefix. |
| `--founder-immutable` | `flag` | `reference/fixed haplotypes, closest` | Freeze founder updates. Important for STITCH-parity diagnostics. |


## 3. Minimal Runs

The examples below are written as shell commands. In JupyterLab, prefix shell commands with `!` or use `%%bash` cells.


In [ ]:
%%bash
# Minimal read-aware diploid run on the synthetic data.
# Remove the leading "echo" when you are ready to execute.
echo stitchv2 run \
  --samples benchmark_runs/synth_5mb_2k_0p1x/samples.parquet \
  --positions benchmark_runs/synth_5mb_2k_0p1x/positions.parquet \
  --chromosome chrSynthetic \
  --output-dir benchmark_runs/tutorial_minimal \
  --n-founders 8 \
  --em-iterations 5 \
  --block-size 1000 \
  --hmm-backend jax \
  --fragment-coupling-model stitch_parity \
  --write-genotype-posteriors \
  --write-genotype-calls


### 3.1 Python API Run

The Python API gives access to features that are awkward in a CLI, especially custom founder panels and pedigree matrices.


In [ ]:
from stitchv2 import PipelineConfig, StitchPipeline

RUN_PIPELINE_EXAMPLE = False
if RUN_PIPELINE_EXAMPLE and DATA_DIR.exists():
    samples = pd.read_parquet(DATA_DIR / 'samples.parquet').head(8).copy()
    cfg = PipelineConfig(
        chromosome='chrSynthetic',
        positions_path=DATA_DIR / 'positions.parquet',
        chromosome_start=None,
        chromosome_end=None,
        output_dir=OUT_BASE / 'python_api_run',
        n_founders=8,
        em_iterations=2,
        block_size=250,
        hmm_backend='jax',
        jax_sample_batch_size=4,
        read_mode='read_stream',
        read_stream_backend='auto',
        use_fragment_likelihood=True,
        fragment_likelihood_mode='augment',
        fragment_coupling_model='stitch_parity',
        write_genotype_posteriors=True,
        write_genotype_calls=True,
        write_support_mask=True,
        calibrate_genotype_posteriors=True,
    )
    pipeline = StitchPipeline(cfg)
    pipeline.prepare_inputs(samples)


## 4. Input Workflow Examples

### 4.1 STITCH-Compatible Text Inputs to STITCHV2 Tables

Synthetic datasets in this repo include STITCH-compatible files:

```text
bamlist.txt
sample_names.txt
pos.txt
```

STITCHV2 prefers `samples.parquet` and `positions.parquet`. This cell shows how to convert the STITCH text layout into STITCHV2 tables.


In [ ]:
def stitch_text_to_stitchv2_tables(data_dir: Path, generation: float = 10.0):
    bam_paths = pd.read_csv(data_dir / 'bamlist.txt', header=None, names=['bam_path'])
    sample_names_path = data_dir / 'sample_names.txt'
    if sample_names_path.exists():
        sample_ids = pd.read_csv(sample_names_path, header=None, names=['sample_id'])
    else:
        sample_ids = pd.DataFrame({'sample_id': [f'sample_{i}' for i in range(len(bam_paths))]})
    samples = pd.concat([sample_ids, bam_paths], axis=1)
    samples['generation'] = float(generation)
    positions = pd.read_csv(data_dir / 'pos.txt', sep=r'\s+', header=None, names=['CHR', 'POS', 'REF', 'ALT'])
    return samples, positions

if DATA_DIR.exists():
    samples_from_stitch, positions_from_stitch = stitch_text_to_stitchv2_tables(DATA_DIR)
    display(samples_from_stitch.head())
    display(positions_from_stitch.head())


### 4.2 Global Microarray PLINK Evidence

Use `--microarray-plink` when one PLINK BED/BIM/FAM prefix contains hard genotype calls for many samples. STITCHV2 aligns PLINK samples by IID to `sample_id`, aligns variants by chromosome/POS, and injects hard-call evidence.

If PLINK contains samples absent from `samples.parquet`, STITCHV2 adds them by default with empty `bam_path`; use `--no-microarray-add-samples` to disable this.


In [ ]:
%%bash
# Example only; replace /data/arrays/all_samples with a real PLINK prefix.
echo stitchv2 run \
  --samples samples.parquet \
  --positions positions.parquet \
  --chromosome chr1 \
  --output-dir out_array_augmented \
  --n-founders 8 \
  --microarray-plink /data/arrays/all_samples \
  --microarray-generation-default 10 \
  --microarray-hard-call-weight 80 \
  --write-genotype-posteriors \
  --write-genotype-calls


### 4.3 Per-Sample PLINK Evidence with `plink_path`

Use `samples.parquet: plink_path` when different samples come from different array batches, or several rows share the same PLINK file. STITCHV2 loads each unique prefix once and extracts only matching sample IDs.


In [ ]:
samples_per_sample_plink = pd.DataFrame({
    'sample_id': ['animal_001', 'animal_002', 'animal_003'],
    'bam_path': ['/data/bams/animal_001.bam', '', '/data/bams/animal_003.bam'],
    'generation': [10.0, 10.0, 10.0],
    'plink_path': ['/data/arrays/batch_A', '/data/arrays/batch_A', '/data/arrays/batch_B'],
})
samples_per_sample_plink


### 4.4 Founder VCF and Founder PLINK Examples

Founder VCF should be indexed if random access is needed by upstream tools. STITCHV2 founder loading uses the provided variant table for alignment.


In [ ]:
%%bash
# Founder VCF example.
echo stitchv2 run \
  --samples samples.parquet \
  --positions positions.parquet \
  --chromosome chr1 \
  --output-dir out_founder_vcf \
  --n-founders 8 \
  --founder-vcf /data/founders/founders.vcf.gz \
  --founder-immutable \
  --hmm-backend jax

# Founder PLINK example.
echo stitchv2 run \
  --samples samples.parquet \
  --positions positions.parquet \
  --chromosome chr1 \
  --output-dir out_founder_plink \
  --n-founders 8 \
  --founder-plink /data/founders/founder_panel \
  --founder-immutable \
  --hmm-backend jax


## 5. Ploidy, Sex Chromosomes, and Mixed Haploid/Diploid Runs

STITCHV2 supports any integer ploidy `>= 0`.

Important behavior:

- `--ploidy 0`: all genotypes become missing without HMM computation.
- `--ploidy 1`: haploid/pseudo-haploid path.
- `--ploidy 2`: diploid fast path.
- `--ploidy >= 3`: generic unordered founder-count state HMM.
- `--ploidy-males` and `--ploidy-females`: per-sample ploidy from `samples.sex`.

For mixed groups, STITCHV2 runs each positive ploidy group and merges outputs. Samples with ploidy 0 receive missing dosage/GP/GT.


In [ ]:
%%bash
# chrX-like: male haploid, female diploid.
echo stitchv2 run \
  --samples samples_with_sex.parquet \
  --positions chrX_positions.parquet \
  --chromosome chrX \
  --output-dir out_chrX \
  --n-founders 8 \
  --ploidy-males 1 \
  --ploidy-females 2 \
  --write-genotype-posteriors \
  --write-genotype-calls

# chrY-like: male haploid, female absent/missing.
echo stitchv2 run \
  --samples samples_with_sex.parquet \
  --positions chrY_positions.parquet \
  --chromosome chrY \
  --output-dir out_chrY \
  --n-founders 8 \
  --ploidy-males 1 \
  --ploidy-females 0 \
  --write-genotype-posteriors \
  --write-genotype-calls


## 6. Resource Scaling: Samples, SNPs, Founders, Ploidy, and Outputs

The main cost drivers are:

| Driver | Scaling intuition | Practical control |
|---|---|---|
| Samples `N` | Almost linear for evidence/output; HMM batches can reduce memory. | `--jax-sample-batch-size`, `--dask-sample-batch-size` |
| SNPs per block `M` | HMM memory grows roughly with block size. | `--block-size`, `--chr-start`, `--chr-end` |
| Founders `K` | Diploid states are roughly `K^2`; generic ploidy states are `comb(K + P - 1, P)`. | `--n-founders` |
| Ploidy `P` | P=1/2 use fast paths; P>=3 uses generic count states. | `--ploidy`, sex ploidy flags |
| Genotype posterior output | Adds `N * M * (P+1)` floats. | `--write-genotype-posteriors` |
| Haplotype output | Adds `N * M * K` floats. | `--write-haplotype-probabilities` |
| Full transitions | Very expensive for state x state matrices. | keep compact transitions unless needed |
| Reads/fragments | Depends on coverage/read length/fragment coupling. | `--read-mode`, memmap, IO workers |

Rule of thumb:

```text
emission/posterior memory ~ samples_per_task * variants_per_block * state_count * 4 bytes * scratch_multiplier
```

For diploid fast path, `state_count ≈ K*K`. For generic ploidy P, `state_count = comb(K + P - 1, P)`.


In [ ]:
from math import comb

def state_count(k: int, ploidy: int, fast_diploid: bool = True) -> int:
    if ploidy <= 0:
        return 1
    if ploidy == 1:
        return k
    if ploidy == 2 and fast_diploid:
        return k * k
    return comb(k + ploidy - 1, ploidy)

for p in [1, 2, 3, 4, 6]:
    print(f'K=8, P={p}: states={state_count(8, p, fast_diploid=(p == 2))}')


### 6.1 JAX vs Dask

Use serial JAX when one block/sample-batch fits memory and the machine is already saturated.

Use Dask when you want:

- live dashboard diagnostics,
- performance reports,
- coarse scheduling over blocks/sample batches/ploidy groups,
- memory-bounded chromosome-scale runs,
- multi-worker or future multi-GPU scaling.

Dask should schedule **coarse** tasks. Avoid tiny blocks; scheduler overhead can dominate.


In [ ]:
%%bash
# Dask-orchestrated JAX example with dashboard/report.
echo stitchv2 run \
  --samples benchmark_runs/synth_5mb_2k_0p1x/samples.parquet \
  --positions benchmark_runs/synth_5mb_2k_0p1x/positions.parquet \
  --chromosome chrSynthetic \
  --output-dir benchmark_runs/tutorial_dask \
  --n-founders 8 \
  --hmm-backend jax \
  --executor dask \
  --dask-n-workers 2 \
  --dask-threads-per-worker 1 \
  --dask-sample-batch-size 12 \
  --dask-dashboard-address 127.0.0.1:8786 \
  --dask-performance-report dask_report.html \
  --dask-task-stream dask_task_stream.json \
  --write-genotype-posteriors \
  --write-genotype-calls


## 7. Output Files and Schemas

STITCHV2 writes block-partitioned parquet datasets under `output_dir`. Each `block=000000.parquet` contains all samples for a contiguous variant block.

Typical layout:

```text
out/
  samples.parquet
  positions.parquet
  founders.parquet
  stage_timings.json
  memory_profile_summary.json
  run_summary.json
  dosage/block=000000.parquet
  genotype_posteriors/block=000000.parquet
  genotype_calls/block=000000.parquet
  support_mask/block=000000.parquet
  transitions/block=000000.parquet
  recombination/block=000000.parquet
  founder_updates/block=000000.parquet
  haplotype_probabilities/block=000000.parquet
```


### 7.1 Core Output Schema

| Dataset | Columns | Notes |
|---|---|---|
| `dosage/` | `sample_id`, `chromosome`, `position`, `dosage`, `block_id` | Dosage is alt allele count, 0..ploidy; NaN if missing. |
| `genotype_posteriors/` | `sample_id`, `chromosome`, `position`, `genotype_posterior`, `block_id` | Fixed-size list length `P+1` for alt allele count 0..P. Diploid length 3. |
| `genotype_calls/` | `sample_id`, `chromosome`, `position`, `genotype_call`, `block_id` | Integer alt allele count; `-1` means no-call. |
| `support_mask/` | `sample_id`, `chromosome`, `position`, `has_supporting_read`, `block_id` | Whether direct read/array evidence supports that sample-site. |
| `transitions/` | `sample_id`, `chromosome`, `position`, `switch_probability`, `stay_probability`, `offdiag_probability`, `block_id` | Compact transition summary. |
| `recombination/` | `chromosome`, `position`, `recombination_rate`, `block_id` | Recombination/rate proxy used for transitions. |
| `founder_updates/` | `chromosome`, `position`, `founder`, `alt_prob`, `block_id` | Updated founder alt probabilities. |
| `haplotype_probabilities/` | `sample_id`, `chromosome`, `position`, `hap_dosage`, `hap_probability`, `block_id` | Fixed-size list length K; optional. |
| `dask_run_summary.json` | JSON | Dask dashboard URL, chunk plan, task diagnostics, report paths. |
| `stage_timings.json` | JSON list | Per-block read/HMM/calibration/write timing and RSS fields. |


In [ ]:
# Inspect output schemas from an existing run if present.
example_run = REPO / 'benchmark_runs' / 'stitch_stitchv2_dask_2026-04-30' / 'stitchv2'
if example_run.exists():
    import pyarrow.parquet as pq
    for dataset in ['dosage', 'genotype_posteriors', 'genotype_calls', 'transitions', 'founder_updates']:
        files = sorted((example_run / dataset).glob('block=*.parquet'))
        if files:
            print('
', dataset, files[0].name)
            print(pq.read_schema(files[0]))


### 7.2 Combine Chunked Outputs

Use `stitchv2 combine` to create one parquet file per dataset and a lazy xarray view backed by Dask.


In [ ]:
%%bash
# Combine all standard output datasets.
echo stitchv2 combine \
  --run-output-dir benchmark_runs/tutorial_minimal \
  --output-dir benchmark_runs/tutorial_minimal/combined

# Combine one explicit parquet directory.
echo stitchv2 combine \
  --run-output-dir benchmark_runs/tutorial_minimal \
  --input-dir benchmark_runs/tutorial_minimal/dosage \
  --output-file benchmark_runs/tutorial_minimal/dosage.parquet


In [ ]:
# Python API for lazy combined outputs.
RUN_COMBINE_EXAMPLE = False
if RUN_COMBINE_EXAMPLE:
    from stitchv2.output import combine_pipeline_outputs
    xds = combine_pipeline_outputs(OUT_BASE / 'python_api_run')
    print(xds)
    print(xds.attrs['combine_summary'].keys())


## 8. Native Parquet/Zarr Output and Explicit BCF Interoperability

STITCHV2 native outputs are parquet because parquet is efficient for block-wise writing and analytics. Hard calls and categorical columns remain in Parquet with dictionary/RLE/bit-packing-friendly encodings. Floating outputs such as dosage, GP, transitions, recombination rates, and founder probabilities can also be stored in xarray/Zarr.

VCF export is not allowed in STITCHV2. BCF export exists only for interoperability and WILL slow down I/O compared with Parquet/Zarr.


In [ ]:
%%bash
# Combine native Parquet chunks and write floating outputs to xarray/Zarr.
echo stitchv2 combine \
  --run-output-dir benchmark_runs/tutorial_minimal \
  --output-dir benchmark_runs/tutorial_minimal/combined \
  --write-zarr

# Export BCF only if an external tool requires it.
echo stitchv2 export-bcf \
  --run-output-dir benchmark_runs/tutorial_minimal \
  --output-bcf benchmark_runs/tutorial_minimal/stitchv2.chrSynthetic.bcf \
  --chromosome chrSynthetic


### 8.1 Last-Resort BCF Conversion

Use this only when a downstream tool cannot read Parquet/Zarr or BCF directly. This conversion is outside the primary STITCHV2 output path.

```bash
bcftools view benchmark_runs/tutorial_minimal/stitchv2.chrSynthetic.bcf -Oz   -o benchmark_runs/tutorial_minimal/stitchv2.chrSynthetic.from_bcf.vcf.gz
bcftools index -t benchmark_runs/tutorial_minimal/stitchv2.chrSynthetic.from_bcf.vcf.gz
```


### 8.2 Export a Simple Dosage Matrix for Non-PLINK Tools

Sometimes a GWAS pipeline wants a sample-by-SNP dosage matrix. You can pivot the parquet dosage output directly.


In [ ]:
RUN_DOSAGE_MATRIX_EXAMPLE = False
if RUN_DOSAGE_MATRIX_EXAMPLE:
    import pyarrow.dataset as ds
    run_dir = OUT_BASE / 'python_api_run'
    dosage_df = ds.dataset(str(run_dir / 'dosage'), format='parquet').to_table().to_pandas()
    dosage_matrix = dosage_df.pivot(index='sample_id', columns='position', values='dosage')
    dosage_matrix.to_parquet(run_dir / 'dosage_matrix_samples_by_position.parquet')
    dosage_matrix.iloc[:5, :5]


## 9. Cross-Validation and Tuning

`stitchv2 cv` runs fold-based tuning over K/nGen/S-style settings and optional LightGBM post-calibration.

Additional CV-only parameters:

| Parameter | Example | STITCH equivalent | Meaning |
|---|---|---|---|
| `--pseudo-truth` | `pseudo_truth.parquet` | external truth/evaluation file | Table with sample_id, position, genotype or dosage truth. |
| `--k-values` | `6,8,10` | K grid | Founder-state grid. |
| `--ngen-values` | `0.75,1.0,1.25` | nGen grid | Generation/recombination scaling grid. |
| `--s-values` | `2,3` | STITCH smoothing/internal setting, closest | Additional tuning dimension used by harness. |
| `--seeds` | `0,1,2` | seed averaging in scripts | Repeat grid across seeds. |
| `--folds` | `5` | no direct equivalent | Number of CV folds. |
| `--holdout-fraction` | `0.2` | no direct equivalent | Fraction held out per fold. |
| `--lightgbm-post-calibrator` | flag | no direct equivalent | Extra post-calibration model. |


In [ ]:
%%bash
# CV/tuning example.
echo stitchv2 cv \
  --samples samples.parquet \
  --positions positions.parquet \
  --pseudo-truth pseudo_truth.parquet \
  --chromosome chr1 \
  --output-dir out_cv \
  --k-values 6,8,10 \
  --ngen-values 0.75,1.0,1.25 \
  --s-values 2,3 \
  --seeds 0,1,2 \
  --folds 5 \
  --holdout-fraction 0.2 \
  --hmm-backend jax \
  --fragment-coupling-model stitch_parity \
  --lightgbm-post-calibrator


## 10. Benchmarking and Diagnostics

Useful benchmark/report scripts in this repo:

| Script | Use |
|---|---|
| `benchmarks/synthetic_dataset.py` | Generate synthetic reads, truth dosage, founders, STITCH-compatible text inputs. |
| `benchmarks/benchmark_compare.py` | Compare STITCHV2 to original STITCH. |
| `benchmarks/benchmark_stitch_stitchv2_dask_report.py` | Three-way STITCH/STITCHV2/STITCHV2-Dask report with plots. |
| `benchmarks/benchmark_ploidy_modes.py` | Ploidy and sex chromosome scenarios. |
| `benchmarks/benchmark_jax_generic_ploidy_parity.py` | Generic ploidy JAX HMM correctness/runtime. |
| `benchmarks/check_memory_leak.py` | Repeated run memory-growth check. |
| `stitchv2 tune-jax-memory` | Block-size, memmap, and JAX sample-batch tuning. |


In [ ]:
%%bash
# Three-way benchmark with plots and Dask performance report.
echo python benchmarks/benchmark_stitch_stitchv2_dask_report.py \
  --data-dir benchmark_runs/synth_5mb_2k_0p1x \
  --output-dir benchmark_runs/stitch_stitchv2_dask_tutorial \
  --chromosome chrSynthetic \
  --k 8 \
  --iterations 5 \
  --block-size 1000 \
  --dask-dashboard-address 127.0.0.1:8786 \
  --force


## 11. Practical Recipes

### Recipe A: Fast diagnostic run on a chromosome slice

Use `--chr-start/--chr-end`, small `--block-size`, and a subset sample table.

### Recipe B: STITCH-parity diagnostic

Use hard immutable founders, `fragment_coupling_model=stitch_parity`, enough EM iterations, and consider `--no-calibrate-genotype-posteriors` if you want raw model comparison.

### Recipe C: Production low-coverage run

Use `hmm_backend=jax`, `read_stream_backend=auto` or `htslib`, calibrated posteriors, genotype calls, support mask, and tune `block_size`/`jax_sample_batch_size`.

### Recipe D: Chromosome-scale memory-controlled run

Use Dask executor with coarse chunks and performance report:

```bash
stitchv2 run ...   --executor dask   --dask-n-workers 4   --dask-threads-per-worker 1   --dask-target-task-memory-mb 4096   --dask-dashboard-address 127.0.0.1:8786   --dask-performance-report dask_report.html
```


## 12. Troubleshooting Checklist

| Symptom | Likely cause | What to check |
|---|---|---|
| No reads overlap variants | contig mismatch or wrong coordinates | `samtools idxstats`, position `CHR`, `--chromosome` |
| Dask slower than serial | chunks too small or dataset too small | increase block/sample batch; use Dask for larger runs |
| JAX OOM | too many samples/SNPs/states per task | lower `--block-size` or `--jax-sample-batch-size`; enable Dask planner |
| STITCHV2 differs from STITCH | founder representation, calibration, no-call policy, read coupling | use hard immutable founders, stitch parity coupling, comparable call mode |
| Sex chromosome wrong missingness | missing/unclear `samples.sex` | use M/F, male/female, 1/2, XY/XX labels |
| PLINK sample not injected | IID does not match `sample_id` | inspect `.fam` IID column and samples table |
| PLINK variant not injected | BIM chromosome/POS mismatch | inspect `.bim`; align `CHR`, `POS`, REF/ALT conventions |
| VCF/PLINK conversion loses dosage | PLINK1 BED is hard-call only | use PLINK2 PGEN with `dosage=DS` |


## 13. Quick Reference: Original STITCH vs STITCHV2 Command Shape

Original STITCH call shape in R:

```r
STITCH::STITCH(
  tempdir = 'tmp',
  chr = 'chr1',
  bamlist = 'bamlist.txt',
  sampleNames_file = 'sample_names.txt',
  posfile = 'pos.txt',
  outputdir = 'stitch_out/',
  K = 8,
  nGen = 10,
  niterations = 5,
  nCores = 4,
  method = 'diploid',
  output_haplotype_dosages = FALSE
)
```

Equivalent STITCHV2 shape:

```bash
stitchv2 run   --samples samples.parquet   --positions positions.parquet   --chromosome chr1   --output-dir stitchv2_out   --n-founders 8   --em-iterations 5   --hmm-backend jax   --io-workers 4   --ploidy 2   --fragment-coupling-model stitch_parity   --write-genotype-posteriors   --write-genotype-calls
```
